# Chess experiment suite: results and paired comparisons

Set **`EXP_DIR`**, then run all cells. This notebook reads saved suite outputs; it never calls models, reruns experiments, or changes result files. Rerun the loading cell to refresh a running experiment, then rerun the analyses.

The four conditions are single-round consultancy, multi-round consultancy, single-round debate, and multi-round debate. Each proposer/judge pair stays separate. **Matched accuracy** compares only puzzle IDs judged in every selected condition for that pair. **Completed accuracy** uses all completed judgements, counting invalid completed verdicts as incorrect. Unjudged puzzles are reported as missing coverage, not as incorrect judgements.

Correctness follows the suite's saved reference-move labels. Comparisons are paired by proposal and puzzle; they are not equal-compute comparisons. No confidence or significance claims are inferred from these descriptive charts.


In [ ]:
from pathlib import Path
from html import escape
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "core" / "chess").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXP_DIR = REPO_ROOT / "exp" / "chess_research"  # Change this to your suite directory.
SAVE_FIGURES = False  # Optional PNG exports; source results are never rewritten.
FIGURE_DIR = EXP_DIR / "figures" / "suite"

CONDITION_LABELS = {
    "single_consultancy": "Single consultancy",
    "multi_consultancy": "Multi consultancy",
    "single_debate": "Single debate",
    "multi_debate": "Multi debate",
}
COLORS = dict(zip(CONDITION_LABELS, ["#64748b", "#0891b2", "#d97706", "#7c3aed"]))
BUCKETS = ["Low", "Medium", "High"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titlepad": 12})


def finish_figure(fig, name):
    fig.tight_layout()
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIGURE_DIR / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.show()


In [ ]:
def bool_column(series, nullable=False):
    values = series.astype("string").str.strip().str.lower()
    allowed = values.isna() | values.isin(["", "true", "false"])
    if not allowed.all():
        raise ValueError(f"Unexpected boolean values in {series.name}: {values[~allowed].unique()}")
    result = values.map({"true": True, "false": False}).astype("boolean")
    return result if nullable else result.fillna(False).astype(bool)


def load_suite(directory):
    directory = Path(directory).expanduser().resolve()
    manifest_path = directory / "chess_suite.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"No chess_suite.json in {directory}. Select a research_suite root.")
    manifest = json.loads(manifest_path.read_text())
    evaluation_path = directory / "evaluations.csv"
    if not evaluation_path.is_file():
        raise FileNotFoundError("No evaluations.csv yet. Wait for the suite to checkpoint its summaries.")
    actual = pd.read_csv(evaluation_path, keep_default_na=False, dtype={"puzzleId": str, "proposal_id": str, "proposer": str, "judge": str})
    keys = ["condition", "proposer", "judge", "puzzleId"]
    required = {*keys, "complete", "correct", "generated", "proposal_valid", "interaction_valid",
                "verdict_valid", "proposer_correct", "proposal_id", "proposed_move", "judgement"}
    if not required.issubset(actual.columns):
        raise ValueError(f"Missing suite evaluation columns: {sorted(required - set(actual.columns))}")
    if actual.duplicated(keys).any():
        raise ValueError("Duplicate condition/proposer/judge/puzzle rows in evaluations.csv.")
    # Reconstruct planned rows so missing checkpoint rows cannot silently shrink denominators.
    plan = pd.DataFrame([
        dict(condition=condition, proposer=pair["proposer"]["id"], judge=pair["judge"]["id"],
             puzzleId=str(puzzle["puzzleId"]), rating_bucket=puzzle["rating_bucket"])
        for pair in manifest["models"] for condition in manifest["conditions"]
        for puzzle in manifest["puzzles"]
    ])
    check = actual[keys].merge(plan[keys], on=keys, how="left", indicator=True)
    if check._merge.ne("both").any():
        raise ValueError("Evaluation rows do not belong to the pinned suite plan.")
    data = plan.merge(actual.drop(columns=["rating_bucket"], errors="ignore"), on=keys,
                      how="left", validate="one_to_one")
    for field in ["complete", "correct", "generated", "proposal_valid", "interaction_valid", "verdict_valid"]:
        data[field] = bool_column(data[field])
    data["proposer_correct"] = bool_column(data["proposer_correct"], nullable=True)
    for field in ["proposal_id", "proposed_move", "judgement"]:
        data[field] = data[field].fillna("").astype(str)
    done = data[data.complete]
    if (done.proposer_correct.isna() | done.proposal_id.eq("")).any():
        raise ValueError("Completed judgements need a ground-truth label and proposal ID.")
    if (done.verdict_valid & ~done.judgement.isin(["Yes", "No"])).any():
        raise ValueError("A saved valid verdict is not Yes or No.")
    if (~done.generated | ~done.proposal_valid).any():
        raise ValueError("Completed judgements refer to absent or invalid proposals.")
    generated = data[data.generated]
    for field in ["proposal_id", "proposed_move", "proposer_correct"]:
        if generated.groupby(["proposer", "judge", "puzzleId"])[field].nunique().gt(1).any():
            raise ValueError(f"Conditions disagree on {field}; these proposals cannot be compared as matched.")
    data["matched"] = False
    for _, group in data.groupby(["proposer", "judge"], sort=False):
        shared = set.intersection(*[
            set(group.loc[group.condition.eq(condition) & group.complete, "puzzleId"])
            for condition in manifest["conditions"]
        ])
        data.loc[group.index, "matched"] = group.puzzleId.isin(shared)
    costs_path = directory / "costs.csv"
    costs = pd.read_csv(costs_path) if costs_path.is_file() else pd.DataFrame()
    return manifest, data, costs


def summarize(data, conditions, by_bucket=False):
    rows = []
    group_keys = ["proposer", "judge", "condition"] + (["rating_bucket"] if by_bucket else [])
    for key, group in data.groupby(group_keys, sort=False):
        done = group[group.complete]
        matched = group[group.matched & group.complete]
        truth = done.proposer_correct.fillna(False).astype(bool)
        yes = done.verdict_valid & done.judgement.eq("Yes")
        no = done.verdict_valid & done.judgement.eq("No")
        tp, tn = int((truth & yes).sum()), int((~truth & no).sum())
        fp, fn = int((~truth & yes).sum()), int((truth & no).sum())
        positives, negatives = int(truth.sum()), int((~truth).sum())
        rows.append(dict(
            zip(group_keys, key), planned=len(group), completed=len(done),
            coverage=100 * len(done) / len(group), matched_count=len(matched),
            accuracy=100 * done.correct.mean() if len(done) else np.nan,
            matched_accuracy=100 * matched.correct.mean() if len(matched) else np.nan,
            balanced_accuracy=50 * (tp / positives + tn / negatives) if positives and negatives else np.nan,
            missing_transcripts=int((~group.generated).sum()),
            invalid_proposals=int((group.generated & ~group.proposal_valid).sum()),
            invalid_interactions=int((group.generated & ~group.interaction_valid).sum()),
            invalid_verdicts=int((~done.verdict_valid).sum()),
            positives=positives, negatives=negatives, tp=tp, tn=tn, fp=fp, fn=fn,
            false_accept_rate=100 * fp / negatives if negatives else np.nan,
            false_reject_rate=100 * fn / positives if positives else np.nan,
        ))
    result = pd.DataFrame(rows)
    result["condition"] = pd.Categorical(result.condition, conditions, ordered=True)
    return result.sort_values(group_keys).reset_index(drop=True)


def paired_changes(data):
    contrasts = [
        ("multi_consultancy", "single_consultancy"),
        ("single_debate", "single_consultancy"),
        ("multi_debate", "single_debate"),
        ("multi_debate", "multi_consultancy"),
    ]
    rows = []
    for (proposer, judge), group in data.groupby(["proposer", "judge"], sort=True):
        matched = group[group.matched & group.complete]
        wide = matched.pivot(index="puzzleId", columns="condition", values="correct")
        for treatment, baseline in contrasts:
            if {treatment, baseline}.issubset(wide.columns):
                delta = wide[treatment].astype(int) - wide[baseline].astype(int)
                rows.append(dict(proposer=proposer, judge=judge,
                    comparison=f"{CONDITION_LABELS[treatment]} − {CONDITION_LABELS[baseline]}",
                    puzzles=len(delta), delta_pp=100 * delta.mean(),
                    improved=int(delta.eq(1).sum()), worsened=int(delta.eq(-1).sum()),
                    unchanged=int(delta.eq(0).sum())))
    return pd.DataFrame(rows, columns=["proposer", "judge", "comparison", "puzzles", "delta_pp",
                                       "improved", "worsened", "unchanged"])


In [ ]:
manifest, evaluations, costs = load_suite(EXP_DIR)
conditions = [c for c in CONDITION_LABELS if c in manifest["conditions"]]
pairs = list(evaluations[["proposer", "judge"]].drop_duplicates().itertuples(index=False, name=None))
summary = summarize(evaluations, conditions)

print(f"Suite: {EXP_DIR}")
print(f"{len(manifest['puzzles']):,} puzzles per pair · {len(pairs)} model pair(s) · "
      f"{len(conditions)} condition(s) · {manifest['num_rounds']} rounds in multi-round conditions")
print(f"Completed judgements: {int(evaluations.complete.sum()):,} / {len(evaluations):,} planned")
if not evaluations.complete.any():
    print("No completed judgements yet. Accuracy and error rates remain N/A; recorded costs may already be available.")
print("Summaries are recomputed from evaluations.csv; source files are read without modification.")
display(summary[["proposer", "judge", "condition", "planned", "completed", "coverage",
                 "matched_count", "matched_accuracy", "accuracy", "balanced_accuracy"]]
        .style.format({c: "{:.1f}%" for c in ["coverage", "matched_accuracy", "accuracy", "balanced_accuracy"]}, na_rep="N/A"))


## Accuracy and coverage

The primary bars use the **same puzzle cohort across every selected condition** for each model pair. Hollow circles show accuracy over all completed judgements, which can use a different cohort during partial runs. Invalid completed verdicts count as incorrect in both. Coverage shows completed/planned judgements; invalid-opening and missing-transcript counts below can overlap other diagnostics and should not be added together.


In [ ]:
fig, axes = plt.subplots(len(pairs), 2, figsize=(13, 4 * len(pairs)), squeeze=False)
for row_index, (proposer, judge) in enumerate(pairs):
    group = summary[summary.proposer.eq(proposer) & summary.judge.eq(judge)].set_index("condition").reindex(conditions)
    x = np.arange(len(conditions))
    ax, coverage_ax = axes[row_index]
    ax.bar(x, group.matched_accuracy, color=[COLORS[c] for c in conditions], label="Matched cohort")
    ax.scatter(x, group.accuracy, marker="o", s=65, facecolors="white", edgecolors="#111827", zorder=3,
               label="All completed")
    for index, (_, item) in enumerate(group.iterrows()):
        value = item.matched_accuracy
        label = f"{value:.1f}%\nn={int(item.matched_count)}" if pd.notna(value) else "N/A\nn=0"
        ax.text(index, (value + 2) if pd.notna(value) else 3, label, ha="center", fontsize=9)
    ax.set(ylim=(0, 125), yticks=[0, 20, 40, 60, 80, 100], ylabel="Judge accuracy (%)", title=f"{proposer} → {judge}")
    ax.legend(fontsize=8, loc="upper left", ncol=2)
    coverage_ax.bar(x, group.completed, color=[COLORS[c] for c in conditions], label="Completed")
    coverage_ax.bar(x, group.planned - group.completed, bottom=group.completed, color="#e2e8f0", label="Unjudged")
    for index, (_, item) in enumerate(group.iterrows()):
        coverage_ax.text(index, item.planned + max(0.2, item.planned * .025),
                         f"{int(item.completed)}/{int(item.planned)}", ha="center", fontsize=9)
    coverage_ax.set(ylabel="Planned judgements", title="Coverage", ylim=(0, max(1, group.planned.max()) * 1.35))
    coverage_ax.legend(fontsize=8, ncol=2)
    for axis in (ax, coverage_ax):
        axis.set_xticks(x, [CONDITION_LABELS[c].replace(" ", "\n", 1) for c in conditions])
finish_figure(fig, "accuracy_and_coverage")
display(summary[["proposer", "judge", "condition", "missing_transcripts", "invalid_proposals",
                 "invalid_interactions", "invalid_verdicts"]])


## What changed for the same puzzles?

Each comparison uses the common completed cohort across **all selected conditions**. A positive change means the first condition judged more proposals correctly. “Improved” counts puzzles that changed from an incorrect to a correct judgement; “worsened” counts the reverse. Percentage-point differences are descriptive, not statistical significance tests.


In [ ]:
changes = paired_changes(evaluations)
if changes.empty:
    print("No matched comparisons available yet, or the selected subset contains no comparison pair.")
else:
    display(changes.style.format({"delta_pp": "{:+.1f}"}))
    fig, axes = plt.subplots(len(pairs), 1, figsize=(11, max(3, 3.6 * len(pairs))), squeeze=False)
    for ax, (proposer, judge) in zip(axes[:, 0], pairs):
        group = changes[changes.proposer.eq(proposer) & changes.judge.eq(judge)]
        if group.empty:
            ax.text(.5, .5, "No matched comparisons yet", ha="center", transform=ax.transAxes)
            ax.set_axis_off()
            continue
        ax.barh(group.comparison, group.delta_pp, color=["#059669" if x >= 0 else "#dc2626" for x in group.delta_pp])
        ax.scatter(group.delta_pp, np.arange(len(group)), color="#334155", s=24, zorder=3)
        limit = max(5, group.delta_pp.abs().max() + 5)
        ax.set_xlim(-limit, limit)
        for position, value in enumerate(group.delta_pp):
            ax.text(value + limit * .025, position, f"{value:+.1f} pp", va="center", fontsize=9)
        ax.axvline(0, color="#475569", linewidth=1)
        ax.set(xlabel="Change in matched judge accuracy (percentage points)", title=f"{proposer} → {judge}")
        ax.invert_yaxis()
    finish_figure(fig, "paired_accuracy_changes")


## False accepts, false rejects, and class balance

**False accept rate** = accepting an incorrect proposal / all completed judgements of incorrect proposals. **False reject rate** = rejecting a correct proposal / all completed judgements of correct proposals. Invalid verdicts remain in these class denominators but are neither accepts nor rejects; they are counted separately. Thus these rates need not complement a valid-decision rate to 100%.

Balanced accuracy is the mean of the correct-accept and correct-reject rates. A missing ground-truth class gives **N/A**, not zero. These diagnostics use all completed judgements; see the matched-cohort comparison above before comparing partial conditions.


In [ ]:
error_columns = ["proposer", "judge", "condition", "positives", "negatives", "tp", "tn", "fp", "fn",
                 "invalid_verdicts", "false_accept_rate", "false_reject_rate", "balanced_accuracy"]
display(summary[error_columns].style.format({c: "{:.1f}%" for c in
        ["false_accept_rate", "false_reject_rate", "balanced_accuracy"]}, na_rep="N/A"))
fig, axes = plt.subplots(len(pairs), 1, figsize=(11, 3.8 * len(pairs)), squeeze=False)
for ax, (proposer, judge) in zip(axes[:, 0], pairs):
    group = summary[summary.proposer.eq(proposer) & summary.judge.eq(judge)].set_index("condition").reindex(conditions)
    x = np.arange(len(conditions))
    for offset, field, label, color in [(-.18, "false_accept_rate", "False accept / incorrect proposals", "#dc2626"),
                                         (.18, "false_reject_rate", "False reject / correct proposals", "#d97706")]:
        ax.bar(x + offset, group[field], width=.34, label=label, color=color)
        for index, value in enumerate(group[field]):
            ax.text(index + offset, value + 2 if pd.notna(value) else 2,
                    f"{value:.1f}%" if pd.notna(value) else "N/A", ha="center", fontsize=8)
    ax.set(ylim=(0, 125), yticks=[0, 20, 40, 60, 80, 100], ylabel="Class-conditioned rate (%)", title=f"{proposer} → {judge}")
    ax.set_xticks(x, [CONDITION_LABELS[c] for c in conditions])
    ax.legend(fontsize=8, ncol=2)
finish_figure(fig, "false_accepts_and_rejects")


## Matched accuracy by difficulty

Only puzzles judged in every selected condition are included. Low: rating <1300; Medium: 1300–2300 inclusive; High: >2300. Bucket counts beneath the axis are the shared cohort sizes, not the originally planned sample sizes. Empty buckets have no plotted point.


In [ ]:
bucket_summary = summarize(evaluations, conditions, by_bucket=True)
fig, axes = plt.subplots(len(pairs), 1, figsize=(10, 4 * len(pairs)), squeeze=False)
for ax, (proposer, judge) in zip(axes[:, 0], pairs):
    subset = bucket_summary[bucket_summary.proposer.eq(proposer) & bucket_summary.judge.eq(judge)]
    for condition in conditions:
        group = subset[subset.condition.eq(condition)].set_index("rating_bucket").reindex(BUCKETS)
        ax.plot(np.arange(3), group.matched_accuracy, marker="o", linewidth=2,
                label=CONDITION_LABELS[condition], color=COLORS[condition])
    cohort = evaluations[evaluations.proposer.eq(proposer) & evaluations.judge.eq(judge) & evaluations.matched]
    counts = cohort.drop_duplicates("puzzleId").rating_bucket.value_counts()
    ax.set_xticks(np.arange(3), [f"{bucket}\n(n={int(counts.get(bucket, 0))})" for bucket in BUCKETS])
    ax.set(ylim=(0, 105), ylabel="Matched judge accuracy (%)", title=f"{proposer} → {judge}")
    ax.legend(fontsize=8)
finish_figure(fig, "matched_accuracy_by_difficulty")
display(bucket_summary[["proposer", "judge", "condition", "rating_bucket", "matched_count", "matched_accuracy"]]
        .style.format({"matched_accuracy": "{:.1f}%"}, na_rep="N/A"))


## Recorded cost and usage

`costs.csv` counts each functional response cache once, including successful calls from failed interactions. Reused openings and transcript prefixes are **not** billed again in this table. Costs with unknown prices are explicitly unpriced; the known subtotal is not a complete total if any are present. Provider retries/errors without returned usage may incur additional unrecorded costs. Costs are by model, not allocated to conditions, since work is shared between conditions.


In [ ]:
if costs.empty:
    print("No recorded response-cache costs yet.")
else:
    required_costs = ["cost", "unknown_cost_calls", "input_tokens", "output_tokens", "api_calls"]
    for field in required_costs:
        costs[field] = pd.to_numeric(costs[field], errors="raise")
    display(costs.style.format({"cost": "${:.6f}", "input_tokens": "{:,.0f}",
                                "output_tokens": "{:,.0f}", "api_calls": "{:,.0f}",
                                "unknown_cost_calls": "{:,.0f}"}))
    unknown = int(costs.unknown_cost_calls.sum())
    print(f"Known recorded cost: ${costs.cost.sum():.6f}; {int(costs.api_calls.sum()):,} cached responses."
          + (f" Plus {unknown:,} responses with unknown cost." if unknown else ""))
    fig, axes = plt.subplots(1, 2, figsize=(12, max(3.5, .65 * len(costs))))
    axes[0].barh(costs.model, costs.cost, color="#0891b2")
    axes[0].set(xlabel="Known recorded cost (USD)", title="Unique cached responses")
    positions = np.arange(len(costs))
    axes[1].barh(positions, costs.input_tokens, color="#94a3b8", label="Input")
    axes[1].barh(positions, costs.output_tokens, left=costs.input_tokens, color="#7c3aed", label="Output")
    axes[1].set_yticks(positions, costs.model)
    axes[1].set(xlabel="Recorded tokens", title="Token usage")
    axes[1].legend()
    finish_figure(fig, "recorded_cost_and_tokens")


## Inspect a puzzle across all conditions

Select a model pair and puzzle ID, then click **Show puzzle**. The viewer displays the original board, locked proposal, public interaction, and final judge response. It prefers the exact input pinned in each judge manifest, with the saved public interaction as a fallback. Raw tool traces and private reasoning are not displayed. The reference move is shown for human analysis only; it was not supplied to the agents.

If widgets are unavailable, call `show_suite_puzzle("puzzle_id", proposer, judge)` after running the helper cell. Refresh the loading cell before inspecting new checkpoints.


In [ ]:
import chess
import chess.svg
from core.chess.protocols.interaction import format_transcript, public_turn


def read_puzzle_rows(path, puzzle_id):
    columns = {"puzzleId", "model", "proposer", "model_response", "transcript", "tool_trace", "complete", "error"}
    frame = pd.read_csv(path, keep_default_na=False, dtype={"puzzleId": str},
                        usecols=lambda column: column in columns)
    return frame[frame.puzzleId.eq(str(puzzle_id))]


def suite_puzzle_html(puzzle_id, proposer=None, judge=None):
    proposer, judge = (proposer, judge) if proposer is not None and judge is not None else pairs[0]
    matches = [p for p in manifest["puzzles"] if str(p["puzzleId"]) == str(puzzle_id)]
    if not matches:
        return f"<p>Unknown puzzle ID: {escape(str(puzzle_id))}</p>"
    puzzle = matches[0]
    board = chess.Board(puzzle["FEN"])
    reference = board.san(board.parse_uci(puzzle["moves"].split()[0]))
    report = [f"<h3>{escape(str(puzzle_id))} — {escape(proposer)} → {escape(judge)}</h3>",
              chess.svg.board(board=board, size=340),
              f"<p>Rating: {puzzle['rating']} · Reference first move: <b>{escape(reference)}</b></p>"]
    selected = evaluations[evaluations.puzzleId.eq(str(puzzle_id)) & evaluations.proposer.eq(proposer)
                           & evaluations.judge.eq(judge)]
    for condition in conditions:
        record = selected[selected.condition.eq(condition)]
        if record.empty:
            continue
        row = record.iloc[0]
        folder = EXP_DIR / "conditions" / condition
        public_text, origin = "", "No public interaction saved yet."
        pinned = folder / "judgements" / "chess_judge_run.json"
        if pinned.is_file():
            sources = json.loads(pinned.read_text()).get("sources", [])
            sources = [s for s in sources if str(s.get("puzzleId")) == str(puzzle_id)
                       and s.get("proposer") == proposer and s.get("target_judge", judge) == judge]
            if len(sources) == 1:
                public_text, origin = sources[0]["model_response"], "Exact public transcript pinned for judging"
        mode = condition.split("_", 1)[1]
        if not public_text:
            for path in sorted((folder / mode).glob("*/puzzles.csv")):
                frame = pd.read_csv(path, keep_default_na=False, dtype={"puzzleId": str},
                                    usecols=lambda c: c in {"puzzleId", "model", "target_judge", "transcript"})
                found = frame[frame.puzzleId.eq(str(puzzle_id)) & frame.model.eq(proposer) & frame.target_judge.eq(judge)]
                if len(found) == 1:
                    public_text = format_transcript(json.loads(found.iloc[0].transcript))
                    origin = "Saved public interaction (not yet pinned for judging)"
                    break
        judge_text, error = "No final judge response saved yet.", ""
        for path in sorted((folder / "judgements" / "judge").glob("*/puzzles.csv")):
            found = read_puzzle_rows(path, puzzle_id)
            found = found[found.proposer.eq(proposer) & found.model.eq(judge)]
            if len(found) == 1:
                saved = found.iloc[0]
                trace = json.loads(saved.get("tool_trace") or "[]")
                _, final = public_turn(trace)
                responses = [entry for entry in trace if "completion" in entry]
                if not responses or responses[-1].get("tool_calls"):
                    final = ""
                judge_text = final or "No final public judge response."
                error = str(saved.get("error", ""))
                break
        status = ("Correct judgement" if row.correct else "Incorrect judgement") if row.complete else "Unjudged"
        report.extend([
            f"<h4>{escape(CONDITION_LABELS[condition])} — {status}</h4>",
            f"<p>Locked move: <b>{escape(row.proposed_move or 'unavailable')}</b> · Verdict: {escape(row.judgement or 'unavailable')}</p>",
            f"<p><em>{escape(origin)}</em></p>",
            '<pre style="white-space:pre-wrap;max-height:560px;overflow:auto;border:1px solid #cbd5e1;padding:12px">'
            + escape(public_text or "No public interaction saved yet.") + '</pre>',
            '<p><b>Final judge response</b></p><pre style="white-space:pre-wrap">' + escape(judge_text) + '</pre>',
            f"<p>Error: {escape(error)}</p>" if error else "",
        ])
    return "".join(report)


def show_suite_puzzle(puzzle_id, proposer=None, judge=None):
    display(HTML(suite_puzzle_html(str(puzzle_id), proposer, judge)))


In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    print("Widgets unavailable. Call show_suite_puzzle(puzzle_id, proposer, judge).")
else:
    pair_input = widgets.Dropdown(options=[(f"{p} → {j}", (p, j)) for p, j in pairs],
                                  description="Model pair:", layout=widgets.Layout(width="95%"))
    puzzle_input = widgets.Combobox(options=sorted(evaluations.puzzleId.unique()),
                                    description="Puzzle ID:", ensure_option=True)
    puzzle_button = widgets.Button(description="Show puzzle", button_style="primary")
    puzzle_output = widgets.HTML()

    def _show_suite_puzzle(_=None):
        if puzzle_input.value:
            puzzle_output.value = suite_puzzle_html(puzzle_input.value, *pair_input.value)

    puzzle_button.on_click(_show_suite_puzzle)
    display(widgets.VBox([pair_input, widgets.HBox([puzzle_input, puzzle_button]), puzzle_output]))
